# Final Chatbot: IT Helpdesk Chatbot for NU Students

This notebook builds the text feature-extraction step for an IT Helpdesk chatbot aimed at NU students. It covers Bag of Words and TF-IDF vectorization on the training dataset, then uses the resulting features to train a simple intent-classification chatbot that can respond to common IT support questions (e.g. password resets, wifi/LMS access issues).

In [ ]:
import pandas as pd
import json
import random
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression

DATA_PATH = "."

In [ ]:
# 1. Load the dataset
df = pd.read_csv(f"{DATA_PATH}/training_data.csv")

print("Dataset shape:", df.shape)
print("Number of unique intents:", df["intent"].nunique())
df.head()

Dataset shape: (509, 2)
Number of unique intents: 41


,text,intent
0,Hi,greeting
1,How are you?,greeting
2,Is anyone there?,greeting
3,Hello,greeting
4,Good day,greeting


In [ ]:
#handles slang queries

normalization_dict = {
    "abt": "about", "u": "you", "ur": "your", "pls": "please", "plz": "please",
    "thx": "thanks", "tnx": "thanks", "info": "information", "san": "saan",
    "kelan": "kailan", "d2": "dito", "b4": "before", "asap": "as soon as possible",
    "yr": "your", "msg": "message", "acc": "account", "reg": "registration",
    "wla": "wala", "pano": "paano", "cge": "sige",
    "gud": "good", "gud am": "good morning", "gud pm": "good afternoon"
}

def normalize_text(text):
    text = text.lower()
    words = text.split()
    normalized_words = [normalization_dict.get(w, w) for w in words]
    return " ".join(normalized_words)

df["text"] = df["text"].apply(normalize_text)
df.head()

,text,intent
0,hi,greeting
1,how are you?,greeting
2,is anyone there?,greeting
3,hello,greeting
4,good day,greeting


In [ ]:
#Bag of Words Model

count_vectorizer = CountVectorizer()
X_counts = count_vectorizer.fit_transform(df["text"])

print("Vocabulary:")
print(count_vectorizer.vocabulary_)

print("\nMatrix shape:", X_counts.shape)

bow_df = pd.DataFrame(X_counts.toarray(), columns=count_vectorizer.get_feature_names_out())
print("\nFirst five rows:")
bow_df.head()

Vocabulary:
{'hi': 164, 'how': 172, 'are': 27, 'you': 425, 'is': 186, 'anyone': 24, 'there': 372, 'hello': 160, 'good': 146, 'day': 91, 'what': 409, 'up': 396, 'ya': 423, 'heyy': 163, 'whatsup': 411, 'po': 301, 'kumusta': 202, 'magandang': 225, 'araw': 26, 'uy': 398, 'pwede': 310, 'magtanong': 229, 'cya': 89, 'see': 338, 'bye': 56, 'later': 208, 'goodbye': 147, 'am': 16, 'leaving': 210, 'have': 156, 'talk': 363, 'to': 383, 'ttyl': 387, 'got': 148, 'go': 145, 'gtg': 151, 'sige': 345, 'salamat': 327, 'ok': 274, 'thank': 369, 'aalis': 0, 'na': 253, 'ako': 14, 'paalam': 287, 'name': 255, 'your': 426, 'do': 100, 'called': 59, 'should': 344, 'call': 58, 'whats': 410, 'who': 417, 'this': 377, 'chatting': 69, 'taking': 362, 'ano': 20, 'pangalan': 293, 'mo': 248, 'sino': 346, 'ka': 192, 'meron': 245, 'bang': 39, 'timing': 380, 'of': 269, 'college': 76, 'working': 422, 'days': 92, 'when': 413, 'guys': 153, 'open': 280, 'hours': 171, 'operation': 281, 'the': 371, 'about': 1, 'on': 278, 'saturday'

,aalis,about,ac,access,account,active,activities,activity,add,address,...,why,wifi,will,work,working,ya,year,you,your,yung
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
#TF-IDF

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df["text"])

print("Vocabulary:")
print(tfidf_vectorizer.vocabulary_)

print("\nMatrix shape:", X_tfidf.shape)

tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
print("\nFirst five rows:")
tfidf_df.head()

Vocabulary:
{'hi': 164, 'how': 172, 'are': 27, 'you': 425, 'is': 186, 'anyone': 24, 'there': 372, 'hello': 160, 'good': 146, 'day': 91, 'what': 409, 'up': 396, 'ya': 423, 'heyy': 163, 'whatsup': 411, 'po': 301, 'kumusta': 202, 'magandang': 225, 'araw': 26, 'uy': 398, 'pwede': 310, 'magtanong': 229, 'cya': 89, 'see': 338, 'bye': 56, 'later': 208, 'goodbye': 147, 'am': 16, 'leaving': 210, 'have': 156, 'talk': 363, 'to': 383, 'ttyl': 387, 'got': 148, 'go': 145, 'gtg': 151, 'sige': 345, 'salamat': 327, 'ok': 274, 'thank': 369, 'aalis': 0, 'na': 253, 'ako': 14, 'paalam': 287, 'name': 255, 'your': 426, 'do': 100, 'called': 59, 'should': 344, 'call': 58, 'whats': 410, 'who': 417, 'this': 377, 'chatting': 69, 'taking': 362, 'ano': 20, 'pangalan': 293, 'mo': 248, 'sino': 346, 'ka': 192, 'meron': 245, 'bang': 39, 'timing': 380, 'of': 269, 'college': 76, 'working': 422, 'days': 92, 'when': 413, 'guys': 153, 'open': 280, 'hours': 171, 'operation': 281, 'the': 371, 'about': 1, 'on': 278, 'saturday'

,aalis,about,ac,access,account,active,activities,activity,add,address,...,why,wifi,will,work,working,ya,year,you,your,yung
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.576273,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0


In [ ]:
sorted_vocab = sorted(count_vectorizer.vocabulary_.keys())
print("Sample of vocabulary (first 20 words):")
print(sorted_vocab[:20])

Sample of vocabulary (first 20 words):
['aalis', 'about', 'ac', 'access', 'account', 'active', 'activities', 'activity', 'add', 'address', 'admision', 'admission', 'against', 'ai', 'ako', 'allotment', 'am', 'an', 'and', 'ang']


#Testing the Chatbot

In [ ]:
#uses TF-IDF
model = LogisticRegression(max_iter=1000)
model.fit(X_tfidf, df["intent"])

print("Model trained on", X_tfidf.shape[0], "examples across", df["intent"].nunique(), "intents")

Model trained on 509 examples across 41 intents


In [ ]:
with open(f"{DATA_PATH}/intents_merged.json") as f:
    intents_data = json.load(f)

#looks up possible responses
response_lookup = {intent["tag"]: intent["responses"] for intent in intents_data["intents"]}

In [ ]:
MIN_GAP = 0.03

while True:
    user_input = input("You: ")
    if user_input.lower() in ["quit", "exit"]:
        print("Bot: Bye! Ingat.")
        break

    cleaned_input = normalize_text(user_input)
    input_vector = tfidf_vectorizer.transform([cleaned_input])

    probabilities = model.predict_proba(input_vector)[0]
    sorted_indices = probabilities.argsort()[::-1]
    best_index = sorted_indices[0]
    second_index = sorted_indices[1]

    best_intent = model.classes_[best_index]
    confidence = probabilities[best_index]
    gap = confidence - probabilities[second_index]

    if gap < MIN_GAP:
        print("Bot: Sorry, I don't understand. Can you rephrase that?")
    else:
        reply = random.choice(response_lookup[best_intent])
        print(f"Bot: {reply}")
        print(f"      (matched intent: {best_intent}, confidence: {confidence:.2f}, gap: {gap:.2f})")

You: timetable?
Bot: Sorry, I don't understand. Can you rephrase that?
You: quit
Bot: Bye! Ingat.
